# PXF, HDFS и Parquet — 30 заданий

PXF выполняется рядом с segment hosts и даёт Greenplum параллельный доступ к HDFS. Все writable пути ограничены `/data/raw/m_razhin`; исходные Метрика и MOEX только читаются.

## Результаты обучения

После **PXF, HDFS и Parquet** вы должны объяснять физическое выполнение на coordinator/segments, связывать logical SQL с Motion/I/O/skew, выбирать дизайн по workload и доказывать решение измерениями.

## Ментальная модель

PXF запускает fragment work рядом с сегментами и преобразует HDFS/Parquet в строки Greenplum. Projection и predicate pushdown определяют, сколько данных покинет HDFS.

```text
client → coordinator (parse/optimize)
              │ dispatch slices
       ┌──────┼──────┐
       ▼      ▼      ▼
    segment segment segment
       └── Motion/interconnect ──┘
              │
              ▼
          coordinator
```
Coordinator не должен становиться местом обработки всех строк. Хороший план оставляет
scan/aggregate на сегментах и перемещает только необходимое.

## Данные и grain

Yandex Metrica и MOEX Parquet в HDFS. Полные схемы находятся в `data-catalog`. Общие external/raw объекты читаются, учебные результаты создаются только в `m_razhin`.

## Инженерный алгоритм

1. Назовите grain и ключ. 2. Оцените объём/cardinality. 3. Выберите distribution/storage/partition. 4. Предскажите Motion и I/O. 5. Создайте минимальный объект. 6. ANALYZE. 7. Снимите EXPLAIN и сегментные метрики. 8. Сверьте результат.

Сверьте PXF profile и LOCATION, типы с Parquet schema, узкую projection, partition filter и число строк с Hive/Spark.

## Типичные ошибки

- Переносить правила PostgreSQL без учёта MPP.
- Выбирать distribution key только по высокой cardinality.
- Путать partitioning с distribution.
- Считать Broadcast всегда плохим, а Redistribute всегда допустимым.
- Сравнивать время единственного запуска без rows/Motion/I/O.
- Создавать external object с путём, доступным Windows, но не сегментам.

## Вопросы для самопроверки

1. Где физически лежит строка? 2. Какие slices выполнят сегменты? 3. Что и сколько передаёт Motion? 4. Как проявится skew? 5. Что произойдёт при повторной загрузке? 6. Как доказать результат из независимого источника?

In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 150
%sql postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex

## 1. Что такое PXF

Platform Extension Framework состоит из Greenplum extension и PXF server на segment hosts. External table хранит URI/profile/options. Во время запроса сегменты получают fragments и обращаются к PXF server, который использует Hadoop client.

## 2. Profile

Profile выбирает connector и формат, например `hdfs:parquet`. Он определяет reader/writer и допустимые options. Неверный profile — ошибка конфигурации, а не SQL-типов.

## 3. Fragment

Fragment — единица внешней работы: файл, блок или логическая часть источника. PXF распределяет fragments между сегментами. Один файл не всегда равен одному fragment, но большое число маленьких файлов почти всегда создаёт planning/open overhead.

## 4. Parquet

Parquet — колоночный self-describing формат с row groups, column chunks, статистикой и compression. PXF сопоставляет физические поля с колонками external table. Порядок/имена и типы зависят от настроек reader и файла.

## 5. Projection pushdown

Если SQL выбирает несколько колонок, Parquet reader может не читать остальные chunks. Выигрыш особенно велик на широкой таблице. `SELECT *`, функции над многими колонками и последующая ненужная проекция увеличивают I/O.

## 6. Predicate pushdown

Часть фильтра передаётся PXF/reader и позволяет пропустить row groups/files. В плане ищите PXF filter. Сложные функции, несовместимые cast и некоторые выражения остаются фильтром Greenplum после чтения.

## 7. Partitioned paths

HDFS-каталоги часто имеют вид `key=value`. Это физическая организация, но не автоматически колонка Greenplum. Hive может добавлять partition column из metastore; прямой PXF path требует явного проектирования.

## 8. Readable external

CREATE не проверяет весь набор и не копирует данные. Ошибки path, schema или type могут проявиться на SELECT. External object не имеет обычной Greenplum distribution policy хранения; распределяется работа чтения.

## 9. Writable external

INSERT создаёт новые файлы. Обычно это append, не overwrite и не транзакционная замена каталога. Не пытайтесь читать ещё не созданный путь до первой записи: некоторые слои кэшируют отрицательный результат.

## 10. Schema evolution

Добавление nullable поля в новые Parquet-файлы может быть совместимо, но смешанный каталог нужно тестировать. Переименование, изменение типа и перестановка при positional mapping опасны. Schema contract хранится вместе с pipeline.

## 11. Типы

Parquet INT64 может соответствовать bigint/timestamp в зависимости logical annotation. Decimal требует precision/scale. Date и timestamp нуждаются в проверке timezone semantics. Строка, объявленная bigint, даст runtime conversion error.

## 12. Small files

Тысячи маленьких файлов ухудшают list/status/open, увеличивают fragments и нагрузку NameNode. Compaction читает набор и пишет меньше крупных файлов в новый каталог, затем consumer переключается после проверки.

## 13. External или internal

External хорош для raw/редкого чтения и обмена. Внутренняя AO column даёт статистику Greenplum, контролируемое distribution, локальные scan и предсказуемые JOIN. Решение зависит от частоты, SLA и стоимости повторного HDFS-чтения.

## 14. Диагностика

Разделяйте уровни: SQL DDL, extension, PXF server 5888, Hadoop config, NameNode/DataNode, path/permissions, file format/schema. `connection refused`, `FileNotFound` и type mismatch требуют разных действий.

## 15. Reconciliation

Сравнивайте HDFS manifest (files/bytes), external rows, accepted/rejected и internal rows/checksum. Count без checksum не выявляет подмену значений; checksum без count может скрыть особенности агрегации.

## 16. Порядок работы

1. Проверить PXF status. 2. Проверить HDFS path. 3. Создать readable external. 4. LIMIT/profile. 5. Count/schema quality. 6. Проверить pushdown. 7. При необходимости записать новый каталог. 8. Создать readable readback. 9. Reconcile. 10. Материализовать target.

### Задание 1. `m_razhin.gpx_01_config`

**Что сделать:** Создайте VIEW параметров PXF/HDFS стенда.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

NameNode и server profile уже настроены.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_01_config здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',1);

### Задание 2. `m_razhin.gpx_02_metrica_ext`

**Что сделать:** Исследуйте существующую dds external Метрики и создайте metadata VIEW.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Читайте pg_exttable, не пересоздавайте source.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_02_metrica_ext здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',2);

### Задание 3. `m_razhin.gpx_03_metrica_count`

**Что сделать:** Зафиксируйте count и диапазон дат PXF-источника.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Внешний count обращается к HDFS.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_03_metrica_count здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',3);

### Задание 4. `m_razhin.gpx_04_metrica_profile`

**Что сделать:** Профилируйте страны, города, IP и даты.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Этот профиль понадобится для физической модели.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_04_metrica_profile здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',4);

### Задание 5. `m_razhin.gpx_05_projection`

**Что сделать:** Сравните чтение 4 колонок и SELECT *.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Parquet читает только нужные column chunks.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_05_projection здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',5);

### Задание 6. `m_razhin.gpx_06_predicate`

**Что сделать:** Сравните фильтр по дате с полным чтением.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Проверьте PXF filter в плане.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_06_predicate здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',6);

### Задание 7. `m_razhin.gpx_07_fragments`

**Что сделать:** Создайте VIEW распределения прочитанных строк по gp_segment_id.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

PXF назначает fragments сегментам.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_07_fragments здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',7);

### Задание 8. `m_razhin.gpx_08_file_layout`

**Что сделать:** Создайте VIEW количества HDFS файлов/партиций источника.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Метаданные можно получить через подготовленный manifest.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_08_file_layout здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',8);

### Задание 9. `m_razhin.gpx_09_schema_map`

**Что сделать:** Создайте VIEW Greenplum↔Parquet типов Метрики.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Особенно даты, bigint и nullable.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_09_schema_map здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',9);

### Задание 10. `m_razhin.gpx_10_null_profile`

**Что сделать:** Профилируйте NULL важных колонок.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Schema совместимость включает nullability.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_10_null_profile здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',10);

## Уровень 2 — writable Parquet и round-trip

### Задание 11. `m_razhin.gpx_11_countries_write`

**Что сделать:** Создайте writable PXF Parquet для countries.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Путь только внутри /data/raw/m_razhin.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_11_countries_write здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',11);

### Задание 12. `m_razhin.gpx_12_countries_insert`

**Что сделать:** Запишите 173 страны в HDFS и сохраните audit count.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Сначала writable, потом readable.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_12_countries_insert здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',12);

### Задание 13. `m_razhin.gpx_13_countries_read`

**Что сделать:** Создайте readable PXF над созданными Parquet-файлами.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Используйте файловый wildcard.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_13_countries_read здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',13);

### Задание 14. `m_razhin.gpx_14_roundtrip`

**Что сделать:** Сверьте count и checksum countries round-trip.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Порядок строк не гарантирован.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_14_roundtrip здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',14);

### Задание 15. `m_razhin.gpx_15_append`

**Что сделать:** Добавьте второй batch и покажите append semantics.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Writable PXF не делает overwrite.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_15_append здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',15);

### Задание 16. `m_razhin.gpx_16_partition_path`

**Что сделать:** Запишите данные в путь вида load_date=YYYY-MM-DD.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Каталог может кодировать partition value.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_16_partition_path здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',16);

### Задание 17. `m_razhin.gpx_17_multi_path`

**Что сделать:** Создайте readable table над несколькими partition paths.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Wildcard должен выбирать ожидаемые файлы.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_17_multi_path здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',17);

### Задание 18. `m_razhin.gpx_18_path_column`

**Что сделать:** Добавьте partition value в результат безопасным способом.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Не путайте физическую колонку и имя каталога.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_18_path_column здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',18);

### Задание 19. `m_razhin.gpx_19_small_files`

**Что сделать:** Создайте VIEW оценки small-file problem.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Много маленьких fragments создаёт overhead.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_19_small_files здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',19);

### Задание 20. `m_razhin.gpx_20_compaction`

**Что сделать:** Выполните учебную compaction в новый каталог.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Не перезаписывайте читаемый source на месте.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_20_compaction здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',20);

## Уровень 3 — MOEX, диагностика и архитектурный выбор

### Задание 21. `m_razhin.gpx_21_moex_ext`

**Что сделать:** Исследуйте external MOEX Parquet и его LOCATION.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Путь уже партиционирован по SECID/date.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_21_moex_ext здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',21);

### Задание 22. `m_razhin.gpx_22_moex_filter`

**Что сделать:** Прочитайте один ticker/day и измерьте строки/fragments.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Фильтр должен совпасть с путём и колонками.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_22_moex_filter здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',22);

### Задание 23. `m_razhin.gpx_23_moex_projection`

**Что сделать:** Сравните узкую и широкую проекцию сделок.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Витрине нужны не все source fields.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_23_moex_projection здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',23);

### Задание 24. `m_razhin.gpx_24_type_mismatch`

**Что сделать:** Воспроизведите безопасную ошибку несовместимого типа.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Сохраните SQLSTATE и сообщение.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_24_type_mismatch здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',24);

### Задание 25. `m_razhin.gpx_25_missing_path`

**Что сделать:** Диагностируйте отсутствующий HDFS path.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Различайте no files и connection failure.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_25_missing_path здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',25);

### Задание 26. `m_razhin.gpx_26_pxf_status`

**Что сделать:** Создайте VIEW статуса PXF обоих segment hosts.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Проверяется порт 5888 на sdw1/sdw2.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_26_pxf_status здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',26);

### Задание 27. `m_razhin.gpx_27_external_vs_internal`

**Что сделать:** Сравните план/время external и внутренней AO copy.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Частое чтение может оправдать материализацию.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_27_external_vs_internal здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',27);

### Задание 28. `m_razhin.gpx_28_pushdown_report`

**Что сделать:** Создайте отчёт predicate/projection pushdown нескольких запросов.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Не каждый SQL-предикат можно передать источнику.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_28_pushdown_report здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',28);

### Задание 29. `m_razhin.gpx_29_reconciliation`

**Что сделать:** Сверьте HDFS manifest, external count и internal target.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

Три уровня должны согласоваться.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_29_reconciliation здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',29);

### Задание 30. `m_razhin.gpx_30_pipeline`

**Что сделать:** Создайте итоговый VIEW PXF pipeline со статусами.

Не изменяйте исходные HDFS-каталоги. Для writable используйте уникальный подкаталог `training/pxf/...` внутри персонального пути.

<details><summary>Подсказка</summary>

HDFS→PXF external→quality→internal→reconciliation.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpx_30_pipeline здесь.

In [ ]:
%%sql
-- Ручная проверка external/readback/manifest.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('pxf_hdfs',30);

## Прогресс

In [ ]:
%%sql
SELECT * FROM greenplum_training.progress WHERE module_name='pxf_hdfs' ORDER BY task_no;